In [ ]:
# # ============================================================
# # IMPORTS AND REPOSITORY SETUP
# # ============================================================

# import os

# repo_url = "https://github.com/tikadatta2005/CSC60904-Deep-Learning-T3.git"
# repo_dir = "CSC60904-Deep-Learning-T3"

# if not os.path.exists(repo_dir):
#     !git clone {repo_url}

# os.chdir(repo_dir)

# !pip install -q scikit-learn seaborn matplotlib pandas

# print("Repository ready.")

In [ ]:
# set the module location
import sys
import os
sys.path.append(os.path.abspath(".."))

In [ ]:
# import deeplearning libraries
import torch
import torch.nn as nn
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader

# Architecture
from modules.architectures.Architecture import Architecture

# Modules for Experimenting
from modules.helper.evaluator import evaluator
from modules.helper.calculate_metrics import calculate_metrics
from modules.helper.charts import gen_line_charts
from modules.helper.trainer import  Trainer
from modules.helper.tester import test
from modules.helper.confusion_matrix_gen import plot_confusion_matrix

# additional
from pathlib import Path

In [ ]:
# Augmented DataLoaders

torch.manual_seed(41)

data_root = "../datasets/final_dataset"
batch_size = 32
img_size = 112

root = Path(data_root)

train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomRotation(10),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.9, 1.1)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
])

transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
])

# Load datasets
train_ds = datasets.ImageFolder(
    root=root / "train",
    transform=train_transform
)

val_ds = datasets.ImageFolder(
    root=root / "val",
    transform=transform
)

test_ds = datasets.ImageFolder(
    root=root / "test",
    transform=transform
)

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True
)

# Get class names
class_names = train_ds.classes

# Display dataset sizes
print(f"Train: {len(train_ds)}")
print(f"Validation: {len(val_ds)}")
print(f"Test: {len(test_ds)}")
print(f"Classes: {class_names}")

In [ ]:
model = Architecture()

model.add(
    # first conv block
    nn.Conv2d(3,8,3,padding=1),
    nn.BatchNorm2d(8),
    nn.ReLU(),
    nn.MaxPool2d(2,2),
    # second conv block
    nn.Conv2d(8,16,3,padding=1),
    nn.BatchNorm2d(16),
    nn.ReLU(),
    nn.MaxPool2d(2,2),
    # third conv block
    nn.Conv2d(16,32,3,padding=1),
    nn.BatchNorm2d(32),
    nn.ReLU(),
    nn.MaxPool2d(2,2),
    # fourt conv block
    nn.Conv2d(32,64,3,padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(2,2),

    # flatten
    nn.Flatten(),

    # fully connected neural network
    nn.Linear(64*7*7,64),

    nn.ReLU(),

    nn.Linear(64, 36)
)

In [ ]:
# Train and save
path = "../documentations/experiments/utsab-aug-lr2e-3-m0-9"

# Using Trainer to train the model
trainer = Trainer(
    model,
    train_loader,
    val_loader,
    optimizer = torch.optim.SGD(
        params=model.parameters(),
        lr=2e-3,
        momentum=0.9,
        weight_decay=3e-5,
    ),
    device="cuda",
    criterion = nn.CrossEntropyLoss()
)

# this is metrucs
metrics = trainer.fit(30, path, 1);

In [ ]:
def find_best_models(df):
    best = df[df["val_accuracy"] == df["val_accuracy"].max()] # combines and remove duplicates
    return best

In [ ]:
find_best_models(metrics)

In [ ]:
gen_line_charts(metrics, path, "training_metrics_graph.png", ["train_", "val_"])